# Run all official collectors

`run_all_collectors()` walks the inventory in wave order and calls the explicit
mapping in `collect.py`. Collectors run **one after another**.

The summary separates:

| Column | Meaning |
| --- | --- |
| `run_status` | This run: `completed` or `failed` |
| `coverage_status` | Stored dataset coverage (`ok`, `blocked`, `partial`, …) |
| `run_error` | Exception text for this run, else null |

A collector that raises is always `run_status=failed`, even if older coverage is still `ok`.
Blocked or partial sources that finish cleanly are `run_status=completed` with the coverage label.

Unique source notes:

- **TN**: statewide handle + privilege tax only; no GGR/AGR
- **CT**: data.ct.gov source CSV (not chart pixels); operator rows only
- **DE sports**: no online/retail split → blocked
- **NY**: weekly cash-basis GGR; negatives kept
- **IL**: join handle to State AGR; online only
- **MD**: Feb 2026 PDF-only gap
- **NV / AR**: combined reports
- **FL / VA**: no official monthly online series
- **MS / MT**: on-premises / location-based, not primary online
- **ME casino**: legal, not yet reporting

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from variant_gaming.collect import inventory_collector_order, run_all_collectors
from variant_gaming.common import project_root
from variant_gaming.consolidate import export_all
from variant_gaming.storage import connect, default_db_path

ROOT = project_root()
scheduled = inventory_collector_order(ROOT)
print("scheduled collectors", len(scheduled))
assert len(scheduled) == 42
scheduled[:8]

scheduled collectors 42


[(1, 'IL', 'online_sports_betting'),
 (1, 'IN', 'online_sports_betting'),
 (1, 'MD', 'online_sports_betting'),
 (1, 'MI', 'online_casino'),
 (1, 'MI', 'online_sports_betting'),
 (1, 'MO', 'online_sports_betting'),
 (1, 'NY', 'online_sports_betting'),
 (1, 'PA', 'online_casino')]

In [2]:
import io
from contextlib import redirect_stdout

# Capture verbose collector logs; show only a short tail after the run.
log_buffer = io.StringIO()
with redirect_stdout(log_buffer):
    summary = run_all_collectors(root=ROOT)

print("summary rows", len(summary))
assert len(summary) == 42
summary[
    [
        "wave",
        "state_code",
        "vertical",
        "run_status",
        "coverage_status",
        "returned_rows",
        "database_rows",
        "run_error",
    ]
]

summary rows 42


,wave,state_code,vertical,run_status,coverage_status,returned_rows,database_rows,run_error
0,1,IL,online_sports_betting,completed,partial,891,891,None
1,1,IN,online_sports_betting,completed,partial,1025,1025,None
2,1,MD,online_sports_betting,completed,partial,351,351,None
3,1,MI,online_casino,completed,ok,1060,1060,None
4,1,MI,online_sports_betting,completed,ok,1060,1060,None
5,1,MO,online_sports_betting,completed,ok,72,72,None
6,1,NY,online_sports_betting,completed,ok,2288,2288,None
7,1,PA,online_casino,completed,ok,936,936,None
8,1,PA,online_sports_betting,completed,ok,1060,1060,None
9,1,TN,online_sports_betting,completed,ok,33,33,None


In [3]:
print("run_status counts")
print(summary.groupby("run_status").size())
print("coverage_status counts")
print(summary.groupby("coverage_status").size())

failed_runs = summary[summary["run_status"] == "failed"]
print("failed runs", len(failed_runs))
failed_runs[["state_code", "vertical", "coverage_status", "run_error"]]

run_status counts
run_status
completed    42
dtype: int64
coverage_status counts
coverage_status
annual_only                       1
blocked                           3
blocked_or_unavailable_export     1
combined_only                     2
legal_not_reporting               1
location_based_mobile             1
not_publicly_available            2
ok                               19
on_premises_only                  1
partial                           4
pdf_only_not_yet_parsed           7
dtype: int64
failed runs 0


,state_code,vertical,coverage_status,run_error


In [4]:
watch = summary[
    summary["coverage_status"].astype(str).isin(
        [
            "blocked",
            "partial",
            "not_publicly_available",
            "combined_only",
            "pdf_only_not_yet_parsed",
            "legal_not_reporting",
            "on_premises_only",
            "location_based_mobile",
            "annual_only",
            "blocked_or_unavailable_export",
            "failed",
        ]
    )
]
watch[["state_code", "vertical", "run_status", "coverage_status", "coverage_reason"]]

,state_code,vertical,run_status,coverage_status,coverage_reason
0,IL,online_sports_betting,completed,partial,January 2020: No Sport Detail CSV for January ...
1,IN,online_sports_betting,completed,partial,2019-07-Revenue.xlsx: no online brand rows; 20...
2,MD,online_sports_betting,completed,partial,February 2026: No Sports Wagering Data Excel d...
12,DE,online_sports_betting,completed,blocked,Official tables split casino sportsbooks vs Sp...
18,AZ,online_sports_betting,completed,blocked,Official reports landing blocked for automated...
19,CO,online_sports_betting,completed,pdf_only_not_yet_parsed,Official monthly Sports Betting Proceeds PDFs ...
20,KS,online_sports_betting,completed,pdf_only_not_yet_parsed,Official monthly detail PDFs separate online/r...
21,KY,online_sports_betting,completed,blocked,Current sports wagering market report is Table...
23,MA,online_sports_betting,completed,pdf_only_not_yet_parsed,Official Category 1/3 sports wagering PDFs are...
24,ME,online_sports_betting,completed,pdf_only_not_yet_parsed,Official operator PDF revenue distributions ar...


In [5]:
log_lines = [line for line in log_buffer.getvalue().splitlines() if line.strip()]
print(f"detailed log lines captured: {len(log_lines)}")
print("--- last 40 lines ---")
for line in log_lines[-40:]:
    print(line)

detailed log lines captured: 805
--- last 40 lines ---
OK NH FY2023: 12 rows
OK NH FY2024: 12 rows
OK NH FY2025: 12 rows
OK NH FY2026: 12 rows
OK NH FY2027: 1 rows
  run=completed coverage=ok: 80 rows returned
Wave 3: OH online_sports_betting
OK OH 2026: 104 rows (2026_Sports_Gaming_Revenue_Report07.xlsx)
OK OH 2025: 179 rows (2025_Sports_Gaming_Revenue_Report12.xlsx)
OK OH 2024: 229 rows (2024_Sports_Gaming_Revenue_Report.xlsx)
OK OH 2023: 233 rows (December_2023_Sports_Gaming_Revenue_Report.xlsx)
  run=completed coverage=ok: 745 rows returned
Wave 3: RI online_casino
  run=completed coverage=pdf_only_not_yet_parsed: 0 rows returned
Wave 3: RI online_sports_betting
  run=completed coverage=pdf_only_not_yet_parsed: 0 rows returned
Wave 3: VA online_sports_betting
  run=completed coverage=not_publicly_available: 0 rows returned
Wave 3: VT online_sports_betting
  run=completed coverage=pdf_only_not_yet_parsed: 0 rows returned
Wave 3: WY online_sports_betting
  run=completed coverage=bloc

In [6]:
paths = export_all(ROOT)
conn = connect(default_db_path(ROOT))
counts = __import__("pandas").read_sql_query(
    "SELECT state_code, vertical, COUNT(*) AS n FROM gaming_results GROUP BY 1, 2 ORDER BY 1, 2",
    conn,
)
coverage_n = conn.execute("SELECT COUNT(*) FROM source_coverage").fetchone()[0]
conn.close()
print("coverage rows", coverage_n)
print("exports", {k: str(v) for k, v in paths.items()})
counts

coverage rows 42
exports {'gaming_results': 'C:\\Users\\Sean\\VscProjects\\researchOS\\data\\processed\\gaming_results.csv', 'state_period_revenue': 'C:\\Users\\Sean\\VscProjects\\researchOS\\data\\processed\\state_period_revenue.csv', 'operator_revenue': 'C:\\Users\\Sean\\VscProjects\\researchOS\\data\\processed\\operator_revenue.csv', 'source_coverage': 'C:\\Users\\Sean\\VscProjects\\researchOS\\data\\processed\\source_coverage.csv'}


,state_code,vertical,n
0,CT,online_casino,223
1,CT,online_sports_betting,174
2,DC,online_sports_betting,62
3,DE,online_casino,612
4,IA,online_sports_betting,1669
5,IL,online_sports_betting,891
6,IN,online_sports_betting,1025
7,LA,online_sports_betting,55
8,MD,online_sports_betting,351
9,MI,online_casino,1060
